## 5.3 RunnableParallel-여러 Runnable을 병렬로 연결하기

- LCEL을 구현하다 보면 Runnable을 병렬로 연결하고 싶을 때가 있다.
- 예를 들어, 사용자가 입력한 주제에 대해 LLM에게 낙관적인 의견과 비관적인 의견을 생성하게 해 보겠다.

In [1]:
# 먼저, model(ChatOpenAI)과 output_parser(StrOutputParser)를 준비
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv("../.env")

model = ChatOpenAI(model="gpt-6-luna")
output_parser = StrOutputParser()

In [2]:
# 다음으로 낙관적인 의견을 생성하는 Chain을 구현
optimistic_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 낙관주의자입니다. 사용자의 입력에 대해 낙관적인 의견을 제공하세요."),
        ("human", "{topic}")
    ]
)
optimistic_chain = optimistic_prompt | model | output_parser

In [3]:
# 비관적인 의견을 생성하는 Chain을 구현
pessimistic_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 비관주의자입니다. 사용자의 입력에 대해 비관적인 의견을 제공하세요."),
        ("human", "{topic}")
    ]
)
pessimistic_chain = pessimistic_prompt | model | output_parser

In [4]:
# RunnableParallel을 사용하여 낙관적인 의견을 생성하는 Chain(optimistic_chain)과 비관적인 의견을 생성하는 Chain(pessimistic_chain)을 병렬로 연결한 Chain을 만들고 실행
import pprint
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    {
        "optimistic_opinion": optimistic_chain,
        "pessimistic_opinion": pessimistic_chain
    }
)
output = parallel_chain.invoke({"topic": "생성 AI의 진화에 관해"})
pprint.pprint(output)

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


{'optimistic_opinion': '생성형 AI의 진화는 **정해진 규칙을 따르던 시스템에서, 맥락을 이해하고 다양한 형태의 콘텐츠를 '
                       '만들어내는 협력 도구로 발전해 온 과정**이라고 볼 수 있습니다.\n'
                       '\n'
                       '- **초기:** 사람이 입력한 규칙과 템플릿에 따라 문장이나 이미지를 생성했습니다. 활용 범위는 '
                       '제한적이었지만 자동화의 가능성을 보여주었습니다.\n'
                       '- **딥러닝의 발전:** 대규모 데이터를 학습하면서 더 자연스러운 문장과 사실적인 이미지를 '
                       '만들기 시작했습니다.\n'
                       '- **대규모 언어 모델의 등장:** 질문과 지시의 맥락을 파악해 글쓰기, 요약, 번역, 코딩 등 '
                       '여러 작업을 수행하게 됐습니다.\n'
                       '- **멀티모달 AI로 확장:** 텍스트뿐 아니라 이미지·음성·영상 등 여러 종류의 정보를 함께 '
                       '이해하고 생성하는 방향으로 발전하고 있습니다.\n'
                       '- **앞으로:** AI가 사람의 목표를 이해해 여러 단계를 수행하고, 개인이나 팀의 업무 방식에 '
                       '맞춰 협력하는 도구로 진화할 가능성이 큽니다.\n'
                       '\n'
                       '이 변화가 특히 기대되는 이유는 AI가 사람을 대체하는 데만 머무르지 않고, 반복적인 작업을 덜어 '
                       '주어 **사람이 창의성·판단·소통에 더 많은 시간을 쓰도록 도울 수 있기 때문**입니다.

### RunnableParallel의 출력을 Runnable의 입력으로 연결하기

In [4]:
# 먼저 낙관적인 의견과 비관적인 의견을 요약하는 프롬프트(synthesize_prompt)를 준비
synthesize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 객관적 AI입니다. 두 가지 의견을 종합하세요."),
        ("human", "낙관적 의견: {optimistic_opinion}\n비관적 의견: {pessimistic_opinion}")
    ]
)

In [6]:
# 낙관적인 의견을 생성하는 Chain(optimistic_chain)과 비관적인 의견을 생성하는 Chain(pessimistic_chain)을 병렬로 연결한 RunnableParallel을 synthesize_prompt, model, output_parser에 연결해 실행
from langchain_core.runnables import RunnableParallel

synthesize_chain = (
    RunnableParallel(
        {
            "optimistic_opinion": optimistic_chain,
            "pessimistic_opinion": pessimistic_chain,
        }
    )
    | synthesize_prompt
    | model
    | output_parser
)

output = synthesize_chain.invoke({"topic": "생성 AI의 진화에 관해"})
print(output)

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


생성형 AI는 규칙 기반의 제한적 생성 도구에서 대규모 언어 모델과 멀티모달 AI를 거쳐, 목표를 세우고 여러 도구를 활용하는 에이전트형 시스템으로 발전하고 있습니다. 이러한 변화는 교육·연구·창작·의료·산업의 생산성과 가능성을 넓힐 수 있으며, 사람의 능력을 보완하는 협력자로 자리 잡을 여지도 있습니다.

그러나 기술의 발전이 곧 사회 전체의 이익으로 이어지는 것은 아닙니다. 허위 정보와 사칭 콘텐츠의 확산, 일자리와 노동 조건의 변화, 편향과 개인정보 침해, 이익의 소수 기업 집중 등이 우려됩니다. 특히 기술이 사회의 제도와 교육, 규제보다 빠르게 발전하면 그 혜택과 부담이 불균등하게 나뉠 수 있습니다.

따라서 생성형 AI의 미래는 기술 성능만으로 결정되지 않습니다. 오류와 피해에 대한 책임, 이익과 비용의 분배, 안전한 활용을 위한 규칙을 어떻게 마련하느냐에 따라 AI는 사람의 역량을 넓히는 동반자가 될 수도, 사회적 불평등과 위험을 키우는 요인이 될 수도 있습니다.


### RunnableParallel 자동 변환

In [ ]:
# 예를 들어 이전 코드는 다음과 같이 작성할 수도 있다.
synthesize_chain = (
    {
        "optimistic_opinion": optimistic_chain,
        "pessimistic_opinion": pessimistic_chain,
    }
    | synthesize_prompt
    | model
    | output_parser
)

### RunnableLambda와의 조합-itemgetter를 사용한 예시

In [7]:
# 예를 들어 {”topic”: “생성 AI의 진화에 관해”}라는 dict에서 itemgetter(”topic”)을 사용해 topic을 추출하는 예는 다음과 같다.
from operator import itemgetter

topic_getter = itemgetter("topic")
topic = topic_getter({"topic": "생성 AI의 진화에 관해"})
print(topic)

생성 AI의 진화에 관해


In [ ]:
# 예를 들어 다음 코드는 값을 추출해 ChatPromptTemplate의 {topic} 부분에 채워 넣는다.
from operator import itemgetter

synthesize_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 객관적 AI입니다. {topic}에 대한 두 가지 의견을 종합하세요."
        ),
        (
            "human",
            "낙관적 의견: {optimistic_opinion}\n비관적 의견: {pessimistic_opinion}"
        )
    ]
)

synthesize_chain = (
    {
        "optimistic_opinion": optimistic_chain,
        "pessimistic_opinion": pessimistic_chain,
        "topic": itemgetter("topic")
    }
    | synthesize_prompt
    | model
    | output_parser
)

output = synthesize_chain.invoke({"topic": "생성 AI의 진화에 관해"})